# 02 - ARMA-GARCH(1,1) Fitting & Rolling Window Innovations

**Project:** ARMA-GARCH Beta Risk Model  
**Author:** Amos Anderson  
**Purpose:** Fit ARMA(1,1)-GARCH(1,1) to log returns for each asset,
diagnose the fit, and extract standardised innovations via a 250-period
rolling estimation window across 1,000 assessment periods.

The innovations output from this notebook are the direct input to
`03_nig_fitting.ipynb` for heavy-tail distribution fitting.

In [1]:
# Imports

import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.io as pio
pio.renderers.default = "iframe"

import warnings
warnings.filterwarnings("ignore")

from src.data_utils import load_processed, save_processed
print("Imports OK")

Imports OK


In [2]:
# Load returns

returns = load_processed("returns.parquet")
TICKERS = list(returns.columns)
print(f"Loaded: {returns.shape[0]} periods × {len(TICKERS)} assets")
print(f"Date range: {returns.index[0].date()} → {returns.index[-1].date()}")
print(f"Tickers: {TICKERS}")

Loaded from C:\Users\amosa\Documents\My Graduate School\SPRING 2026\Courses\AMS603_Risk Measures for Finance and Data Analysis\Projects\arma-garch-beta-risk-model\data\processed\returns.parquet  shape=(5047, 5)
Loaded: 5047 periods × 5 assets
Date range: 2005-01-04 → 2025-12-30
Tickers: ['AAPL', 'EURUSD=X', 'GLD', 'TIP', '^GSPC']


----

## Rolling window design

| Parameter | Value | Rationale |
|-----------|-------|-----------|
| Estimation window | 250 periods | ~1 trading year; project's standard; enough to identify GARCH parameters robustly |
| Assessment window | 1,000 periods | ~4 trading years; provides sufficient exceedances for binomial testing |
| Total periods used | 1,250 | Well within our 5,047 available |
| Step size | 1 day | True out-of-sample: each prediction uses only past data |

The rolling window slides forward one day at a time. At each step:
1. Fit ARMA(1,1)-GARCH(1,1) on the 250-period estimation window
2. Extract the one-step-ahead $\sigma $ forecast
3. Compute standardised innovations: ε_t = r_t / σ_t
4. Store the innovation and $\sigma $ forecast for the prediction date

In [6]:
# Implement src/arma_garch.py

import importlib
import src.arma_garch as _ag
importlib.reload(_ag)
from src.arma_garch import fit_arma_garch, rolling_window_innovations, ljung_box_test
print("src.arma_garch imported OK")

src.arma_garch imported OK


In [7]:
# Single-window diagnostic (S&P 500)
## Before running the full loop, we verify the fit on one window manually.

ticker = "^GSPC"
window_data = returns[ticker].iloc[:250].values

result = fit_arma_garch(window_data)

print(f"Converged: {result.converged}")
print(f"\nFitted parameters:")
for k, v in result.params.items():
    print(f"  {k:20s}: {v:.6f}")

print(f"\nInnovations - mean: {result.innovations.mean():.4f}  "
      f"std: {result.innovations.std():.4f}")
print("Expected: mean ≈ 0, std ≈ 1")

Converged: True

Fitted parameters:
  Const               : 0.035383
  y[1]                : -0.063190
  omega               : 0.037274
  alpha[1]            : 0.105693
  beta[1]             : 0.797522

Innovations - mean: -0.0220  std: 0.9978
Expected: mean ≈ 0, std ≈ 1


In [10]:
# Ljung box test on single window

lb = ljung_box_test(result.innovations)

print("Ljung-Box test on single estimation window (^GSPC, first 250 days)")
print(f"\n  Innovations    - stat: {lb['lb_stat_innov']:.3f}  "
      f"p-value: {lb['lb_pval_innov']:.4f}")
print(f"  Innovations²   - stat: {lb['lb_stat_sq']:.3f}  "
      f"p-value: {lb['lb_pval_sq']:.4f}")
print("\n  p > 0.05 on both implies ARMA-GARCH removed autocorrelation and ARCH effects")
print("  p < 0.05 on innovations² implies residual ARCH effects remain (flag this)")

Ljung-Box test on single estimation window (^GSPC, first 250 days)

  Innovations    - stat: 6.974  p-value: 0.7279
  Innovations²   - stat: 9.321  p-value: 0.5020

  p > 0.05 on both implies ARMA-GARCH removed autocorrelation and ARCH effects
  p < 0.05 on innovations² implies residual ARCH effects remain (flag this)


----

## Full rolling window for all assets

Running 1,000 one-step-ahead predictions per asset.
Each window: fit ARMA(1,1)-GARCH(1,1) on 250 periods -> extract $sigma $ forecast ->
compute innovation for the next day.

**This will take several minutes.** Progress prints every 100 windows.

In [12]:
# Running the rolling window loop

In [11]:
ESTIMATION_WINDOW = 250
ASSESSMENT_WINDOW = 1000

all_results = {}

for ticker in TICKERS:
    print(f"\n{'='*50}")
    print(f"Processing: {ticker}")
    print(f"{'='*50}")

    df = rolling_window_innovations(
        returns[ticker],
        estimation_window=ESTIMATION_WINDOW,
        assessment_window=ASSESSMENT_WINDOW,
    )
    all_results[ticker] = df

print("\nAll assets complete.")


Processing: AAPL
  100/1000 windows complete (3 convergence failures so far)
  200/1000 windows complete (42 convergence failures so far)
  300/1000 windows complete (42 convergence failures so far)
  400/1000 windows complete (42 convergence failures so far)
  500/1000 windows complete (42 convergence failures so far)
  600/1000 windows complete (42 convergence failures so far)
  700/1000 windows complete (42 convergence failures so far)
  800/1000 windows complete (42 convergence failures so far)
  900/1000 windows complete (42 convergence failures so far)
  1000/1000 windows complete (42 convergence failures so far)

Done. 1000 predictions. Convergence failures: 42 (4.2%)

Processing: EURUSD=X
  100/1000 windows complete (0 convergence failures so far)
  200/1000 windows complete (45 convergence failures so far)
  300/1000 windows complete (78 convergence failures so far)
  400/1000 windows complete (78 convergence failures so far)
  500/1000 windows complete (78 convergence failur

In [13]:
# Save innovations

# Save each asset's rolling results
for ticker, df in all_results.items():
    safe_name = ticker.replace("^", "").replace("=", "_")
    save_processed(df, f"rolling_{safe_name}.parquet")

# Also save a combined innovations DataFrame for easy access in notebook 03
innovations_combined = pd.DataFrame({
    ticker: all_results[ticker]["innovation"]
    for ticker in TICKERS
})
save_processed(innovations_combined, "innovations.parquet")

sigma_combined = pd.DataFrame({
    ticker: all_results[ticker]["sigma_forecast"]
    for ticker in TICKERS
})
save_processed(sigma_combined, "sigma_forecasts.parquet")

print("\nSaved: rolling_*.parquet, innovations.parquet, sigma_forecasts.parquet")

Saved to C:\Users\amosa\Documents\My Graduate School\SPRING 2026\Courses\AMS603_Risk Measures for Finance and Data Analysis\Projects\arma-garch-beta-risk-model\data\processed\rolling_AAPL.parquet
Saved to C:\Users\amosa\Documents\My Graduate School\SPRING 2026\Courses\AMS603_Risk Measures for Finance and Data Analysis\Projects\arma-garch-beta-risk-model\data\processed\rolling_EURUSD_X.parquet
Saved to C:\Users\amosa\Documents\My Graduate School\SPRING 2026\Courses\AMS603_Risk Measures for Finance and Data Analysis\Projects\arma-garch-beta-risk-model\data\processed\rolling_GLD.parquet
Saved to C:\Users\amosa\Documents\My Graduate School\SPRING 2026\Courses\AMS603_Risk Measures for Finance and Data Analysis\Projects\arma-garch-beta-risk-model\data\processed\rolling_TIP.parquet
Saved to C:\Users\amosa\Documents\My Graduate School\SPRING 2026\Courses\AMS603_Risk Measures for Finance and Data Analysis\Projects\arma-garch-beta-risk-model\data\processed\rolling_GSPC.parquet
Saved to C:\Users\

In [14]:
# Plot: Conditional volatility

fig = make_subplots(
    rows=len(TICKERS), cols=1,
    shared_xaxes=True,
    subplot_titles=TICKERS,
    vertical_spacing=0.06,
)

colors = px.colors.qualitative.Plotly

for i, ticker in enumerate(TICKERS):
    sigma = all_results[ticker]["sigma_forecast"] * np.sqrt(252)  # annualised

    fig.add_trace(
        go.Scatter(
            x=sigma.index,
            y=sigma.values,
            mode="lines",
            name=ticker,
            line=dict(color=colors[i], width=0.8),
            showlegend=True,
        ),
        row=i+1, col=1,
    )
    fig.update_yaxes(title_text="Ann. vol", row=i+1, col=1)

fig.update_layout(
    title="GARCH(1,1) Conditional Volatility - All Assets (Annualised)",
    height=900,
    template="plotly_white",
    hovermode="x unified",
)
fig.write_html("../outputs/figures/02_conditional_volatility.html")
fig.show()
print("Figure saved to outputs/figures/02_conditional_volatility.html")

Figure saved to outputs/figures/02_conditional_volatility.html


In [15]:
# Standardised Innovations

fig = make_subplots(
    rows=len(TICKERS), cols=1,
    shared_xaxes=True,
    subplot_titles=TICKERS,
    vertical_spacing=0.06,
)

for i, ticker in enumerate(TICKERS):
    innov = all_results[ticker]["innovation"]

    fig.add_trace(
        go.Scatter(
            x=innov.index,
            y=innov.values,
            mode="lines",
            name=ticker,
            line=dict(color=colors[i], width=0.6),
            showlegend=True,
        ),
        row=i+1, col=1,
    )
    fig.add_hline(y=0, line_dash="dash", line_color="black",
                  line_width=0.8, row=i+1, col=1)
    fig.update_yaxes(title_text="Innovation", row=i+1, col=1)

fig.update_layout(
    title="Standardised Innovations After ARMA-GARCH Filtering",
    height=900,
    template="plotly_white",
    hovermode="x unified",
)
fig.write_html("../outputs/figures/02_innovations.html")
fig.show()
print("Figure saved to outputs/figures/02_innovations.html")

Figure saved to outputs/figures/02_innovations.html


In [24]:
# Ljung-box across all assets

lb_rows = []

for ticker in TICKERS:
    innov = all_results[ticker]["innovation"].values
    lb    = ljung_box_test(innov)
    lb_rows.append({
        "Ticker":               ticker,
        "LB stat (innovations)":   round(lb["lb_stat_innov"], 3),
        "p-value (innovations)":   round(lb["lb_pval_innov"], 4),
        "LB stat (innov²)":        round(lb["lb_stat_sq"], 3),
        "p-value (innov²)":        round(lb["lb_pval_sq"], 4),
        "Autocorrelation removed": "✓" if lb["lb_pval_innov"] > 0.05 else "✗",
        "ARCH effects removed":    "✓" if lb["lb_pval_sq"]    > 0.05 else "✗",
    })

lb_df = pd.DataFrame(lb_rows).set_index("Ticker")
print("Ljung-Box Diagnostic Results (lag=10)\n")
display(lb_df)
#print(lb_df.to_string())

Ljung-Box Diagnostic Results (lag=10)



,LB stat (innovations),p-value (innovations),LB stat (innov²),p-value (innov²),Autocorrelation removed,ARCH effects removed
Ticker,,,,,,
AAPL,13.067,0.2199,58.201,0.0000,✓,✗
EURUSD=X,7.264,0.7003,10.225,0.4210,✓,✓
GLD,7.813,0.6471,24.973,0.0054,✓,✗
TIP,21.967,0.0153,17.605,0.0620,✗,✓
^GSPC,9.376,0.4968,41.684,0.0000,✓,✗


In [20]:
# Plot: Innovation distributions vs Gaussian

fig = make_subplots(rows=1, cols=len(TICKERS),
                    subplot_titles=TICKERS,
                    horizontal_spacing=0.05)

x_range = np.linspace(-6, 6, 400)
gaussian = (1 / np.sqrt(2 * np.pi)) * np.exp(-0.5 * x_range**2)

for i, ticker in enumerate(TICKERS):
    innov = all_results[ticker]["innovation"].dropna().values

    fig.add_trace(
        go.Histogram(
            x=innov,
            nbinsx=80,
            histnorm="probability density",
            name=ticker,
            marker_color=colors[i],
            opacity=0.65,
            showlegend=True,
        ),
        row=1, col=i+1,
    )
    fig.add_trace(
        go.Scatter(
            x=x_range,
            y=gaussian,
            mode="lines",
            name="Gaussian" if i == 0 else None,
            showlegend=(i == 0),
            line=dict(color="black", width=1.5),
        ),
        row=1, col=i+1,
    )
    fig.update_xaxes(title_text="Innovation", range=[-6, 6], row=1, col=i+1)

fig.update_layout(
    title="Innovation Distributions vs Standard Gaussian",
    height=420,
    template="plotly_white",
)
fig.write_html("../outputs/figures/02_innovation_distributions.html")
fig.show()
print("Figure saved to outputs/figures/02_innovation_distributions.html")

Figure saved to outputs/figures/02_innovation_distributions.html


In [21]:
# Summary statistics of Innovations

from src.data_utils import summary_statistics

innov_df = pd.DataFrame({
    ticker: all_results[ticker]["innovation"]
    for ticker in TICKERS
})

stats = summary_statistics(innov_df)
print("Innovation Summary Statistics\n")
print(stats.round(4).to_string())
print("\nNote: excess kurtosis >> 0 confirms heavy tails remain after GARCH filtering.")
print("This justifies NIG over Gaussian for the innovation distribution in notebook 03.")

Innovation Summary Statistics

                      AAPL   EURUSD=X        GLD        TIP      ^GSPC
mean (daily)        0.0562    -0.0090     0.0881     0.0208     0.0468
std (daily)         1.1106     1.0465     1.0654     0.9960     1.0501
mean (annual)      14.1734    -2.2643    22.1936     5.2390    11.7824
vol (annual)       17.6305    16.6125    16.9132    15.8110    16.6693
skewness           -0.0823    -0.0460    -0.0865    -0.0738    -0.6245
excess kurtosis     4.8948     2.4707     1.5013     1.6227     2.3754
min                -7.4040    -5.6906    -4.3833    -4.3707    -5.7570
max                 5.9798     5.6464     5.2684     4.9105     3.5591
obs              1000.0000  1000.0000  1000.0000  1000.0000  1000.0000

Note: excess kurtosis >> 0 confirms heavy tails remain after GARCH filtering.
This justifies NIG over Gaussian for the innovation distribution in notebook 03.


In [22]:
# Verification

innov_check = load_processed("innovations.parquet")
sigma_check = load_processed("sigma_forecasts.parquet")

assert innov_check.shape == (ASSESSMENT_WINDOW, len(TICKERS)), \
    f"Shape mismatch: {innov_check.shape}"
assert innov_check.isna().sum().sum() == 0, \
    "NaNs found in innovations"

print(f"Verification PASSED")
print(f"Innovations:      {innov_check.shape}")
print(f"Sigma forecasts:  {sigma_check.shape}")
print(f"Date range:       {innov_check.index[0].date()} to {innov_check.index[-1].date()}")

Loaded from C:\Users\amosa\Documents\My Graduate School\SPRING 2026\Courses\AMS603_Risk Measures for Finance and Data Analysis\Projects\arma-garch-beta-risk-model\data\processed\innovations.parquet  shape=(1000, 5)
Loaded from C:\Users\amosa\Documents\My Graduate School\SPRING 2026\Courses\AMS603_Risk Measures for Finance and Data Analysis\Projects\arma-garch-beta-risk-model\data\processed\sigma_forecasts.parquet  shape=(1000, 5)
Verification PASSED
Innovations:      (1000, 5)
Sigma forecasts:  (1000, 5)
Date range:       2021-11-05 to 2025-12-30


----

## Handoff to notebook 03

Saved to `data/processed/`:

- `innovations.parquet` -- 1,000 $\times $ 5 standardised innovations (ε_t = r_t / σ_t)
- `sigma_forecasts.parquet` -- 1,000 $\times $ 5 one-step-ahead conditional volatilities
- `rolling_{ticker}.parquet` -- full per-asset rolling results including raw returns

**Notebook 03** fits Normal Inverse Gaussian (NIG) and Student T distributions
to these innovations, comparing both against a Gaussian baseline.
The NIG parameters + $\sigma $ forecasts feed directly into VaR/CVaR computation
in notebook 04.